# WC2026 Final, Spain versus Argentina, Pass Network Construction

This notebook builds directed weighted pass networks for Spain and Argentina
from the World Cup 2026 final match event data. It infers pass receivers,
computes average player positions, constructs full match and temporal phase
graphs, and exports extended edgelist files for use in a downstream analysis
notebook.

## Section 1, Setup and Data Loading

Import the required libraries and load the raw event data.

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import os
import json

np.random.seed(42)
plt.style.use('default')

DATA_FILE = '../data/raw/wc2026_spain_vs_argentina_2026-07-19_events.csv'

try:
    if not os.path.exists(DATA_FILE):
        raise FileNotFoundError('Data file not found at ' + DATA_FILE)
    df = pd.read_csv(DATA_FILE, encoding='utf-8')
except Exception as e:
    print('Failed to load the match event data file:', e)
    raise

print('Loaded match event data with shape (rows, columns):', df.shape)
print('Column names available in the dataset:')
print(list(df.columns))

Loaded match event data with shape (rows, columns): (2086, 124)
Column names available in the dataset:
['away_score', 'home_score', 'away_team', 'home_team', 'match_date', 'match_id', 'id', 'eventId', 'minute', 'second', 'teamId', 'x', 'y', 'expandedMinute', 'period', 'type', 'outcomeType', 'qualifiers', 'satisfiedEventsTypes', 'isTouch', 'playerId', 'endX', 'endY', 'relatedEventId', 'relatedPlayerId', 'blockedX', 'blockedY', 'goalMouthZ', 'goalMouthY', 'isShot', 'cardType', 'isGoal', 'team', 'player', 'event', 'outcome', 'period_value', 'period_name', 'qual_Zone', 'qual_PassEndX', 'qual_PassEndY', 'qual_Length', 'qual_Angle', 'qual_Longball', 'qual_Offensive', 'qual_Foul', 'qual_OppositeRelatedEvent', 'qual_Defensive', 'qual_FreekickTaken', 'qual_LayOff', 'qual_ThrowIn', 'qual_Chipped', 'qual_IntentionalAssist', 'qual_ShotAssist', 'qual_KeyPass', 'qual_RegularPlay', 'qual_Assisted', 'qual_RelatedEventId', 'qual_BoxRight', 'qual_LeftFoot', 'qual_LowRight', 'qual_GoalMouthY', 'qual_Goal

## Section 2, Data Cleaning and Filtering

Only in play periods are kept for the pass network construction. PreMatch
and PostGame rows are administrative markers and are removed entirely. The
remaining rows are then sorted into true chronological match order.

In [2]:
period_order = {
    'FirstHalf': 0,
    'SecondHalf': 1,
    'FirstPeriodOfExtraTime': 2,
    'SecondPeriodOfExtraTime': 3
}

valid_periods = list(period_order.keys())

df_filtered = df[df['period_name'].isin(valid_periods)].copy()
print('Number of rows remaining after discarding PreMatch and PostGame rows:', df_filtered.shape[0])

df_filtered['period_sort_key'] = df_filtered['period_name'].map(period_order)

df_sorted = df_filtered.sort_values(
    by=['period_sort_key', 'expandedMinute', 'second', 'eventId'],
    ascending=[True, True, True, True]
).reset_index(drop=True)

print('Total number of rows in df_sorted after chronological sorting:', df_sorted.shape[0])

Number of rows remaining after discarding PreMatch and PostGame rows: 2082
Total number of rows in df_sorted after chronological sorting: 2082


In [3]:
try:
    red_card_rows = df_sorted[(df_sorted['cardType'] == 'SecondYellow') & (df_sorted['team'] == 'Argentina')]
    if len(red_card_rows) == 0:
        raise ValueError('No SecondYellow card event found for Argentina in the filtered data')
    RED_CARD_MINUTE = int(red_card_rows.iloc[0]['expandedMinute'])
except Exception as e:
    print('Failed to identify the Argentina red card event:', e)
    raise

print('Argentina second yellow card, red card, occurred at expanded minute:', RED_CARD_MINUTE)

Argentina second yellow card, red card, occurred at expanded minute: 97


## Section 3, Pass Receiver Inference Algorithm

The dataset does not explicitly encode the receiver of each pass. The
receiver is therefore inferred as the player who performed the next touch
event for the same team in chronological order after the pass. A touch
event is any event where isTouch equals True. If no subsequent touch exists
for the same team within the same period, the pass is discarded, which
naturally handles sequences that end a period.

Two additional edge cases in the raw data are handled explicitly. First, a
small number of touch rows carry no player value, for example a Foul marker
awarded to a team without a named player, so those rows cannot identify a
receiver and the search continues past them. Second, if the inferred
receiver turns out to be the same player as the passer, the pass is
discarded rather than recorded as a self loop. This happens when a stoppage
such as the Foul marker above breaks the true passing sequence and the next
identifiable touch is the same player restarting play, for example taking
the resulting free kick themselves, which is not a genuine passing
connection between two players.

In [4]:
def infer_pass_receiver(df_team):
    df_team = df_team.reset_index(drop=True)
    passes = []
    self_loop_discard_count = 0
    n = len(df_team)
    for i in range(n):
        row = df_team.iloc[i]
        if row['event'] == 'Pass' and row['outcome'] == 'Successful':
            passer_name = row['player']
            origin_x = row['x']
            origin_y = row['y']
            minute = row['expandedMinute']
            period = row['period_name']
            receiver_name = None
            for j in range(i + 1, n):
                next_row = df_team.iloc[j]
                if next_row['period_name'] != period:
                    break
                # A touch row with no recorded player, for example a Foul marker, cannot
                # identify a receiver, so the search continues past it instead of stopping.
                if next_row['isTouch'] and pd.notna(next_row['player']):
                    receiver_name = next_row['player']
                    break
            if receiver_name is not None:
                if receiver_name == passer_name:
                    # A stoppage broke the real passing sequence and the next
                    # identifiable touch is the passer restarting play themselves,
                    # so this is not a genuine pass to a different player.
                    self_loop_discard_count += 1
                    continue
                passes.append((passer_name, receiver_name, origin_x, origin_y, minute, period))
    print('Discarded', self_loop_discard_count, 'pass or passes where the only identifiable next touch belonged to the passer themselves, self loops caused by a stoppage in play.')
    return passes

In [5]:
spain_df = df_sorted[df_sorted['team'] == 'Spain']
argentina_df = df_sorted[df_sorted['team'] == 'Argentina']

print('Inferring pass receivers for Spain:')
spain_passes = infer_pass_receiver(spain_df)
print('Inferring pass receivers for Argentina:')
argentina_passes = infer_pass_receiver(argentina_df)

print('Number of inferred passes with a resolved, distinct receiver for Spain:', len(spain_passes))
print('Number of inferred passes with a resolved, distinct receiver for Argentina:', len(argentina_passes))

Inferring pass receivers for Spain:
Discarded 1 pass or passes where the only identifiable next touch belonged to the passer themselves, self loops caused by a stoppage in play.
Inferring pass receivers for Argentina:


Discarded 0 pass or passes where the only identifiable next touch belonged to the passer themselves, self loops caused by a stoppage in play.
Number of inferred passes with a resolved, distinct receiver for Spain: 792
Number of inferred passes with a resolved, distinct receiver for Argentina: 377


## Section 4, Node Position Computation

Each node, representing a player, is positioned at the average of all their
touch coordinates throughout the match. This average position represents
where the player tended to operate on the pitch and is computed from all
isTouch equal True events for that player, not just passes. Coordinates x
and y range from 0 to 100 in this dataset.

In [6]:
def compute_player_positions(df_team):
    touches = df_team[(df_team['isTouch'] == True) & (df_team['player'].notna())]
    grouped = touches.groupby('player').agg(mean_x=('x', 'mean'), mean_y=('y', 'mean'))
    positions = {player: (row['mean_x'], row['mean_y']) for player, row in grouped.iterrows()}
    return positions

In [7]:
spain_positions = compute_player_positions(spain_df)
argentina_positions = compute_player_positions(argentina_df)

print('Average pitch positions computed for', len(spain_positions), 'Spain players:')
for player, pos in spain_positions.items():
    print(player, ', average x:', round(pos[0], 2), ', average y:', round(pos[1], 2))

print()
print('Average pitch positions computed for', len(argentina_positions), 'Argentina players:')
for player, pos in argentina_positions.items():
    print(player, ', average x:', round(pos[0], 2), ', average y:', round(pos[1], 2))

Average pitch positions computed for 17 Spain players:
Aymeric Laporte , average x: 39.91 , average y: 69.01
Dani Olmo , average x: 61.64 , average y: 43.29
Eric García , average x: 31.85 , average y: 67.23
Fabián Ruiz , average x: 54.75 , average y: 57.93
Ferran Torres , average x: 71.98 , average y: 45.05
Lamine Yamal , average x: 68.36 , average y: 18.62
Marc Cucurella , average x: 55.76 , average y: 85.16
Martín Zubimendi , average x: 45.27 , average y: 49.35
Mikel Merino , average x: 54.22 , average y: 34.66
Mikel Oyarzabal , average x: 59.52 , average y: 42.51
Nico Williams , average x: 76.13 , average y: 76.84
Pau Cubarsí , average x: 38.52 , average y: 30.47
Pedri , average x: 58.57 , average y: 58.44
Pedro Porro , average x: 50.84 , average y: 13.65
Rodri , average x: 51.38 , average y: 45.23
Unai Simón , average x: 13.38 , average y: 52.28
Álex Baena , average x: 74.9 , average y: 71.62

Average pitch positions computed for 17 Argentina players:
Alexis Mac Allister , average 

## Section 5, Graph Construction

Build a directed weighted graph of inferred passes for each team, then
attach the average position node attributes computed in Section 4.

In [8]:
def build_pass_graph(pass_list):
    G = nx.DiGraph()
    for passer, receiver, x, y, minute, period in pass_list:
        if G.has_edge(passer, receiver):
            G[passer][receiver]['weight'] += 1
        else:
            G.add_edge(passer, receiver, weight=1)
    return G

In [9]:
G_spain = build_pass_graph(spain_passes)
G_argentina = build_pass_graph(argentina_passes)

def print_graph_summary(G, team_name):
    total_weight = sum(data['weight'] for _, _, data in G.edges(data=True))
    print('Team:', team_name)
    print('Number of nodes, players involved in completed passing sequences:', G.number_of_nodes())
    print('Number of edges, unique passing connections:', G.number_of_edges())
    print('Total edge weight, total completed passes with a resolved receiver:', total_weight)
    print('Players in the graph:', list(G.nodes()))
    print()

print_graph_summary(G_spain, 'Spain')
print_graph_summary(G_argentina, 'Argentina')

Team: Spain
Number of nodes, players involved in completed passing sequences: 17
Number of edges, unique passing connections: 163
Total edge weight, total completed passes with a resolved receiver: 792
Players in the graph: ['Aymeric Laporte', 'Pau Cubarsí', 'Unai Simón', 'Fabián Ruiz', 'Dani Olmo', 'Mikel Oyarzabal', 'Pedro Porro', 'Álex Baena', 'Marc Cucurella', 'Rodri', 'Lamine Yamal', 'Pedri', 'Ferran Torres', 'Mikel Merino', 'Nico Williams', 'Martín Zubimendi', 'Eric García']

Team: Argentina
Number of nodes, players involved in completed passing sequences: 17
Number of edges, unique passing connections: 147
Total edge weight, total completed passes with a resolved receiver: 377
Players in the graph: ['Lionel Messi', 'Enzo Fernández', 'Rodrigo De Paul', 'Cristian Romero', 'Lisandro Martínez', 'Nicolás Tagliafico', 'Alexis Mac Allister', 'Emiliano Martínez', 'Julián Alvarez', 'Gonzalo Montiel', 'Nico González', 'Nicolás Otamendi', 'Leandro Paredes', 'Facundo Medina', 'Nahuel Molina

In [10]:
spain_self_loops = list(nx.selfloop_edges(G_spain))
argentina_self_loops = list(nx.selfloop_edges(G_argentina))

print('Self loop check, Spain graph:', len(spain_self_loops), 'self loops found,', spain_self_loops)
print('Self loop check, Argentina graph:', len(argentina_self_loops), 'self loops found,', argentina_self_loops)
print('The self loop discard step in infer_pass_receiver removes these before the graph is built, so both counts are expected to be zero.')

Self loop check, Spain graph: 0 self loops found, []
Self loop check, Argentina graph: 0 self loops found, []
The self loop discard step in infer_pass_receiver removes these before the graph is built, so both counts are expected to be zero.


In [11]:
def assign_positions(G, positions, team_name):
    for node in G.nodes():
        if node in positions:
            avg_x, avg_y = positions[node]
            G.nodes[node]['avg_x'] = avg_x
            G.nodes[node]['avg_y'] = avg_y
        else:
            G.nodes[node]['avg_x'] = None
            G.nodes[node]['avg_y'] = None
            print('Warning, no position data found for player', node, 'on team', team_name)

assign_positions(G_spain, spain_positions, 'Spain')
assign_positions(G_argentina, argentina_positions, 'Argentina')

print('Node position attributes assigned for Spain and Argentina graphs.')

Node position attributes assigned for Spain and Argentina graphs.


In [12]:
G_argentina.graph['red_card_minute'] = RED_CARD_MINUTE
G_argentina.graph['red_card_player'] = 'Enzo Fernandez'
G_argentina.graph['note'] = 'Argentina played with 10 players from expandedMinute 97 onward (SecondHalf)'

print('Graph level metadata attached to G_argentina:')
print(json.dumps(G_argentina.graph, indent=2, ensure_ascii=False))

Graph level metadata attached to G_argentina:
{
  "red_card_minute": 97,
  "red_card_player": "Enzo Fernandez",
  "note": "Argentina played with 10 players from expandedMinute 97 onward (SecondHalf)"
}


## Section 6, Temporal Snapshot Construction

Beyond the single static graph, a temporal analysis is possible by splitting
the match into phases. Four temporal phases are defined based on period_name
and expandedMinute.

Phase 1 is FirstHalf, covering minutes 0 to 45 plus stoppage time.
Phase 2 is SecondHalf before the red card, covering minutes 46 up to and
including RED_CARD_MINUTE.
Phase 3 is SecondHalf after the red card, covering minutes RED_CARD_MINUTE
plus 1 up to 90 plus stoppage time.
Phase 4 is ExtraTime, covering FirstPeriodOfExtraTime and
SecondPeriodOfExtraTime combined.

Phase 3 and Phase 4 are the phases where Argentina plays with 10 men, which
is analytically significant for comparing passing behavior under a numerical
disadvantage.

In [13]:
def build_temporal_graphs(pass_list, red_card_minute):
    phase1 = [p for p in pass_list if p[5] == 'FirstHalf']
    phase2 = [p for p in pass_list if p[5] == 'SecondHalf' and p[4] <= red_card_minute]
    phase3 = [p for p in pass_list if p[5] == 'SecondHalf' and p[4] > red_card_minute]
    phase4 = [p for p in pass_list if p[5] in ('FirstPeriodOfExtraTime', 'SecondPeriodOfExtraTime')]

    G_phase1 = build_pass_graph(phase1)
    G_phase2 = build_pass_graph(phase2)
    G_phase3 = build_pass_graph(phase3)
    G_phase4 = build_pass_graph(phase4)

    return {
        'phase1_firsthalf': G_phase1,
        'phase2_before_red': G_phase2,
        'phase3_after_red': G_phase3,
        'phase4_extratime': G_phase4
    }

In [14]:
spain_temporal = build_temporal_graphs(spain_passes, RED_CARD_MINUTE)
argentina_temporal = build_temporal_graphs(argentina_passes, RED_CARD_MINUTE)

def print_temporal_summary(temporal_graphs, team_name):
    print('Temporal phase summary for', team_name)
    for phase_name, G_phase in temporal_graphs.items():
        total_weight = sum(data['weight'] for _, _, data in G_phase.edges(data=True))
        print('Phase:', phase_name, ', edges:', G_phase.number_of_edges(), ', total weight:', total_weight)
    print()

print_temporal_summary(spain_temporal, 'Spain')
print_temporal_summary(argentina_temporal, 'Argentina')

Temporal phase summary for Spain
Phase: phase1_firsthalf , edges: 79 , total weight: 327
Phase: phase2_before_red , edges: 94 , total weight: 263
Phase: phase3_after_red , edges: 13 , total weight: 14
Phase: phase4_extratime , edges: 77 , total weight: 188

Temporal phase summary for Argentina
Phase: phase1_firsthalf , edges: 66 , total weight: 164
Phase: phase2_before_red , edges: 75 , total weight: 118
Phase: phase3_after_red , edges: 0 , total weight: 0
Phase: phase4_extratime , edges: 50 , total weight: 95



### Verification of the phase 3 boundary for Argentina

The Argentina phase 3 after red graph has 0 edges. This is checked directly
against the raw event data below rather than assumed, since an empty phase
could either be a genuine feature of the match or a bug in the phase
boundary logic.

In [15]:
argentina_after_red_window = df_sorted[
    (df_sorted['team'] == 'Argentina') &
    (df_sorted['period_name'] == 'SecondHalf') &
    (df_sorted['expandedMinute'] > RED_CARD_MINUTE)
]

print('Total Argentina event rows in SecondHalf strictly after the red card minute,', RED_CARD_MINUTE, ':', len(argentina_after_red_window))
print('Breakdown of those rows by event type and outcome:')
print(argentina_after_red_window.groupby(['event', 'outcome']).size().to_string())

argentina_after_red_passes = argentina_after_red_window[argentina_after_red_window['event'] == 'Pass']
print()
print('Argentina Pass rows in that window:', len(argentina_after_red_passes))
print('Of those, Successful Pass rows in that window:', (argentina_after_red_passes['outcome'] == 'Successful').sum())
print('Since there are zero Successful Pass rows for Argentina in this window, the empty phase3_after_red graph is a genuine characteristic of this short and chaotic closing stretch of the second half, not an error in the phase boundary logic.')

Total Argentina event rows in SecondHalf strictly after the red card minute, 97 : 7
Breakdown of those rows by event type and outcome:
event          outcome     
BallRecovery   Successful      1
CornerAwarded  Unsuccessful    1
End            Successful      1
Foul           Unsuccessful    1
Interception   Successful      1
Pass           Unsuccessful    1
Save           Successful      1

Argentina Pass rows in that window: 1
Of those, Successful Pass rows in that window: 0
Since there are zero Successful Pass rows for Argentina in this window, the empty phase3_after_red graph is a genuine characteristic of this short and chaotic closing stretch of the second half, not an error in the phase boundary logic.


## Section 7, Edgelist Export

The edgelist format used here is an extended format. Each line contains
source, target, weight, avg_x_source, avg_y_source, avg_x_target,
avg_y_target, separated by spaces. This is a custom weighted edgelist that
carries positional metadata for use in the analysis notebook.

In [16]:
def export_edgelist(G, positions, filepath):
    try:
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write('# source target weight avg_x_source avg_y_source avg_x_target avg_y_target\n')
            for u, v, data in G.edges(data=True):
                weight = data['weight']
                pos_u = positions.get(u)
                pos_v = positions.get(v)
                avg_x_u = pos_u[0] if pos_u is not None else 'NA'
                avg_y_u = pos_u[1] if pos_u is not None else 'NA'
                avg_x_v = pos_v[0] if pos_v is not None else 'NA'
                avg_y_v = pos_v[1] if pos_v is not None else 'NA'
                u_out = u.replace(' ', '_')
                v_out = v.replace(' ', '_')
                f.write(str(u_out) + ' ' + str(v_out) + ' ' + str(weight) + ' ' + str(avg_x_u) + ' ' + str(avg_y_u) + ' ' + str(avg_x_v) + ' ' + str(avg_y_v) + '\n')
        print('Successfully exported edgelist with', G.number_of_edges(), 'edges to', filepath)
    except Exception as e:
        print('Failed to export edgelist to', filepath, ':', e)
        raise

In [17]:
export_edgelist(G_spain, spain_positions, '../data/clean/spain_passes.edgelist')
export_edgelist(G_argentina, argentina_positions, '../data/clean/argentina_passes.edgelist')

Successfully exported edgelist with 163 edges to ../data/clean/spain_passes.edgelist
Successfully exported edgelist with 147 edges to ../data/clean/argentina_passes.edgelist


In [18]:
phase_suffixes = ['phase1_firsthalf', 'phase2_before_red', 'phase3_after_red', 'phase4_extratime']

print('Exporting temporal phase edgelist files for Spain:')
for phase_key in phase_suffixes:
    export_edgelist(spain_temporal[phase_key], spain_positions, '../data/clean/spain_' + phase_key + '.edgelist')

print()
print('Exporting temporal phase edgelist files for Argentina:')
for phase_key in phase_suffixes:
    export_edgelist(argentina_temporal[phase_key], argentina_positions, '../data/clean/argentina_' + phase_key + '.edgelist')

Exporting temporal phase edgelist files for Spain:
Successfully exported edgelist with 79 edges to ../data/clean/spain_phase1_firsthalf.edgelist
Successfully exported edgelist with 94 edges to ../data/clean/spain_phase2_before_red.edgelist
Successfully exported edgelist with 13 edges to ../data/clean/spain_phase3_after_red.edgelist
Successfully exported edgelist with 77 edges to ../data/clean/spain_phase4_extratime.edgelist

Exporting temporal phase edgelist files for Argentina:
Successfully exported edgelist with 66 edges to ../data/clean/argentina_phase1_firsthalf.edgelist
Successfully exported edgelist with 75 edges to ../data/clean/argentina_phase2_before_red.edgelist
Successfully exported edgelist with 0 edges to ../data/clean/argentina_phase3_after_red.edgelist
Successfully exported edgelist with 50 edges to ../data/clean/argentina_phase4_extratime.edgelist


## Section 8, Verification and Summary

Reload the two main edgelist files with NetworkX to confirm that the files
were written correctly and that the weights survive a round trip, then print
a final summary table.

In [19]:
edge_data_spec = (
    ('weight', int),
    ('avg_x_source', str),
    ('avg_y_source', str),
    ('avg_x_target', str),
    ('avg_y_target', str)
)

try:
    G_spain_reloaded = nx.read_edgelist(
        '../data/clean/spain_passes.edgelist',
        comments='#',
        create_using=nx.DiGraph(),
        data=edge_data_spec,
        encoding='utf-8'
    )
    print('Reloaded Spain edgelist, nodes:', G_spain_reloaded.number_of_nodes(), ', edges:', G_spain_reloaded.number_of_edges())
except Exception as e:
    print('Failed to reload the Spain edgelist file:', e)
    raise

try:
    G_argentina_reloaded = nx.read_edgelist(
        '../data/clean/argentina_passes.edgelist',
        comments='#',
        create_using=nx.DiGraph(),
        data=edge_data_spec,
        encoding='utf-8'
    )
    print('Reloaded Argentina edgelist, nodes:', G_argentina_reloaded.number_of_nodes(), ', edges:', G_argentina_reloaded.number_of_edges())
except Exception as e:
    print('Failed to reload the Argentina edgelist file:', e)
    raise

spain_weight_original = sum(data['weight'] for _, _, data in G_spain.edges(data=True))
spain_weight_reloaded = sum(data['weight'] for _, _, data in G_spain_reloaded.edges(data=True))
argentina_weight_original = sum(data['weight'] for _, _, data in G_argentina.edges(data=True))
argentina_weight_reloaded = sum(data['weight'] for _, _, data in G_argentina_reloaded.edges(data=True))

print('Spain total weight preserved after reload:', spain_weight_original == spain_weight_reloaded)
print('Argentina total weight preserved after reload:', argentina_weight_original == argentina_weight_reloaded)

Reloaded Spain edgelist, nodes: 17 , edges: 163


Reloaded Argentina edgelist, nodes: 17 , edges: 147
Spain total weight preserved after reload: True
Argentina total weight preserved after reload: True


In [20]:
summary_data = {
    'Team': ['Spain', 'Argentina'],
    'Total Passes': [len(spain_passes), len(argentina_passes)],
    'Nodes (Players)': [G_spain.number_of_nodes(), G_argentina.number_of_nodes()],
    'Edges (Unique Connections)': [G_spain.number_of_edges(), G_argentina.number_of_edges()],
    'Phases with data': [
        sum(1 for G_phase in spain_temporal.values() if G_phase.number_of_edges() > 0),
        sum(1 for G_phase in argentina_temporal.values() if G_phase.number_of_edges() > 0)
    ],
    'Red card minute': [RED_CARD_MINUTE, RED_CARD_MINUTE]
}

summary_df = pd.DataFrame(summary_data)
print('Final summary table of the pass networks built in this notebook:')
print(summary_df.to_string(index=False))

Final summary table of the pass networks built in this notebook:


     Team  Total Passes  Nodes (Players)  Edges (Unique Connections)  Phases with data  Red card minute
    Spain           792               17                         163                 4               97
Argentina           377               17                         147                 3               97


## Summary

The files exported by this notebook are spain_passes.edgelist and
argentina_passes.edgelist, the full match pass networks for each team, along
with eight temporal phase edgelist files, four per team, covering the first
half, the second half before the red card, the second half after the red
card, and extra time.

Underscores in player names inside the edgelist files should be replaced
back with spaces when displaying results in the analysis notebook.

G_argentina.graph['red_card_minute'] carries the red card annotation, along
with the red card player and a descriptive note, for use in temporal
analysis in the analysis notebook.

Two data edge cases were checked directly against the raw event rows in this
notebook rather than left unexplained. Self loops caused by a stoppage in
play breaking the true passing sequence are discarded during receiver
inference and confirmed absent from both graphs with an explicit
nx.selfloop_edges check. The empty argentina_phase3_after_red graph was
verified against the raw rows to be a genuine feature of a short, chaotic
closing stretch of the second half, since Argentina has zero completed
passes in that window, and not a boundary bug.